# Tabular Parsing

In [4]:
from src.lake_agent.indexing.tabular.vector_store import (
    build_pgvector_store
)

vector_store = build_pgvector_store(table_name='document_index')

In [6]:
query = (
    '''
    Project core nào của AXIOM - iSE có nhiều thành viên hiện tại nhất, không tính số SV mới? Yêu cầu chỉ trả về đúng số thứ tự của project được định nghĩa trong tài liệu. 
    '''
)

results = vector_store.similarity_search_with_score(
    query,
    k=10,
)

for doc, score in results:
    print("score:", score)
    print("record_type:", doc.metadata.get("record_type"))
    print("filename:", doc.metadata.get("filename"))
    print("table_id:", doc.metadata.get("table_id"))
    print("table_name:", doc.metadata.get("table_name"))
    print("sheet_name:", doc.metadata.get("sheet_name"))
    print()
    print(doc.page_content[:800])
    print("-" * 80)

score: 0.45124384493966485
record_type: section
filename: iSE-AXIOM-Internal Intro.pdf
table_id: None
table_name: None
sheet_name: None

Project 5: Platform AXIOM
- Members: Hiếu + Thủy, Vinh, Vũ, Việt, Giáp, Lưu + 1 SV mới
- Objective: Integrate data, workflow generation, execution, validation, and evidence synthesis into a unified platform.
- Focus:
- architecture;
- execution environment;
- data management & pipeline/workflow;
- others
--------------------------------------------------------------------------------
score: 0.47280649288752574
record_type: section
filename: iSE-AXIOM-Internal Intro.pdf
table_id: None
table_name: None
sheet_name: None

Project 7+8: Số hoá + SiFLEX
- Thành viên: Thuỷ, Duy Quân, Minh Quân, Lê Anh Duy, Tuấn Anh + 1 SV mới
- Mục tiêu: Xây dựng pipeline số hóa và khai thác dữ liệu/tài liệu theo hướng có cấu trúc, có truy vết và có khả năng tích hợp vào các hệ thống phân tích.
- Tập trung:
- Dữ liệu thật
- Bài toán siêu lớn
- OCR chuẩn hiệu năng cao
- Tác độ

In [16]:
results[0][0].metadata

{'record_type': 'file',
 'source_id': 'source_ab9528009db4c60c',
 'relative_path': 'archeology/roman_cities.csv',
 'filename': 'roman_cities.csv',
 'file_format': 'csv'}

# OCR

In [15]:
from gradio_client import Client

client = Client("https://fc116af103e4897c8f.gradio.live")

Loaded as API: https://fc116af103e4897c8f.gradio.live/


In [28]:
from gradio_client import handle_file

result = client.predict(
    [
        handle_file("./data/Data-Lake/definitely-100-percent-not-ise-members-image.jpg"),
        # handle_file("./data/Data-Lake/topic_2_page-0004.jpg"),
        # handle_file("./data/Data-Lake/topic_2_page-0004.jpg"),
    ],
    api_name="/ocr"
)

print(result)

["![A large group of young people posing for a photo in front of a thatched-roof building.](beea33802d7f2cc56a0e13313ed802af_1_img.webp)A large group of approximately 40 young people, mostly men, are posing for a group photo outdoors. They are arranged in several rows, with some standing and others kneeling or sitting on a low stone wall. The group is diverse in age and is dressed in casual, everyday clothing such as t-shirts, polo shirts, and jackets. Some individuals are wearing glasses. The background features a traditional building with a steep, thatched roof and a small dark window. Lush green trees with small, round fruits are visible on the left side of the frame. In the foreground, there is a decorative patterned mat on the ground. To the right of the group, there are several cardboard boxes, one of which is labeled 'Vinatissue', and a blue plastic container. The overall atmosphere is cheerful and candid."]


# VLM

In [20]:
from langchain.chat_models import init_chat_model
import os
from dotenv import load_dotenv

load_dotenv()

model = init_chat_model(
    model_provider="openai",
    api_key=os.getenv("OPENAI_API_KEY"),
    base_url=os.getenv("OPENAI_BASE_URL"),
    model=os.getenv("OPENAI_MODEL_NAME"),
    temperature=0,
    )

In [ ]:
import base64
import mimetypes
from langchain_core.messages import HumanMessage

def encode_image(image_path):
    mime_type, _ = mimetypes.guess_type(image_path)
    mime_type = mime_type or "image/jpeg"

    with open(image_path, "rb") as f:
        image_base64 = base64.b64encode(f.read()).decode("utf-8")

    return image_base64, mime_type


image_path = "./data/Data-Lake/scholarship1.png"
image_base64, mime_type = encode_image(image_path)

message = HumanMessage(
    content=[
        {
            "type": "text",
            "text": "...",
        },
        {
            "type": "image",
            "base64": image_base64,
            "mime_type": mime_type,
        },
    ]
)

response = model.invoke([message])

print(response.content)

The image provides information about various scholarship opportunities for the academic year 2022-2023, specifically from the Vietnam National University (ĐHQGHN). Here are the key details:

### Total Scholarships Available:
- **600 scholarships** from various sources.

### Scholarship Details:
1. **Đinh Thiện Lý**
   - Amount: 14,000,000 VND
   - Quantity: 18 scholarships
   - Award Period: 05/2022 & 02/2023

2. **Kumho Asiana**
   - Amount: 2,125,000 VND
   - Quantity: 67 scholarships
   - Award Period: 09/2022 & 03/2023

3. **Lotte, South Korea**
   - Amount: 400 USD
   - Quantity: 15 scholarships
   - Award Period: Not specified

4. **Posco, South Korea**
   - Amount: 1,000 USD
   - Quantity: 18 scholarships
   - Award Period: 09/2022

5. **ADF, South Korea**
   - Amount: 2,000 USD (40 scholarships) / 1,000 USD (12 scholarships)
   - Award Period: 08/2022 & 09/2022

6. **Toshiba, Japan**
   - Amount: 200,000 JPY (14 scholarships) / 100,000 JPY (14 scholarships)
   - Award Period: 0